In [12]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

import pytorch_lightning as pl
from pytorch_lightning.callbacks import (
    ModelCheckpoint, EarlyStopping, LearningRateMonitor, Callback,
)
from pytorch_lightning.loggers import WandbLogger

import wandb
import torchmetrics
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Lightning      : {pl.__version__}")

PyTorch        : 2.7.1+cu118
CUDA available : True
Lightning      : 2.5.1.post0


In [13]:
# ── Dataset ───────────────────────────────────────────────────────────────
DATA_ROOT   = 'classification_dataset'
NUM_CLASSES = 3

# ── DataLoader ────────────────────────────────────────────────────────────
BATCH_SIZE  = 32
NUM_WORKERS = 4

# ── Training schedule ─────────────────────────────────────────────────────
EPOCHS_P1 = 10   # Phase 1: classifier head only, backbone frozen
EPOCHS_P2 = 15   # Phase 2: fine-tune classifier + last backbone stage

# ── Optimiser ─────────────────────────────────────────────────────────────
LR_P1        = 1e-3   # higher LR – backbone is fully frozen
LR_P2        = 1e-5   # very low LR – prevents catastrophic forgetting
WEIGHT_DECAY = 1e-2

# ── WandB ─────────────────────────────────────────────────────────────────
WANDB_API_KEY  = "68591fc7177da3394b90d1c3d2ef0183d85215ab"
WANDB_PROJECT  = "convnext-fish-classification"
WANDB_RUN_NAME = "convnext-tiny-2phase"

In [14]:
train_dir = os.path.join(DATA_ROOT, 'train')
val_dir   = os.path.join(DATA_ROOT, 'valid')

# Use the official pretrained weights' recommended transform pipeline
weights         = models.ConvNeXt_Tiny_Weights.DEFAULT
base_transforms = weights.transforms()

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    base_transforms,   # resize 256, crop 224, ImageNet normalisation
])
val_transforms = base_transforms

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transforms)
val_dataset   = datasets.ImageFolder(root=val_dir,   transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True,
)

CLASS_NAMES = train_dataset.classes
print(f"Classes  : {CLASS_NAMES}")
print(f"Train    : {len(train_dataset):,} images")
print(f"Val      : {len(val_dataset):,} images")

Classes  : ['background', 'fish', 'partial_fish']
Train    : 2,649 images
Val      : 1,618 images


In [15]:
class ConvNextClassifier(pl.LightningModule):
    """
    Two-phase transfer learning wrapper around ConvNeXt Tiny.

    phase=1  →  only the classifier head is trainable  (lr = LR_P1)
    phase=2  →  classifier + last backbone stage unfrozen  (lr = LR_P2)
    """

    def __init__(
        self,
        num_classes: int = 3,
        phase: int = 1,
        lr: float = 1e-3,
        weight_decay: float = 1e-2,
        max_epochs: int = 10,
    ):
        super().__init__()
        self.save_hyperparameters()

        # ── Backbone ────────────────────────────────────────────────────
        backbone_weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        self.model = models.convnext_tiny(weights=backbone_weights)

        for param in self.model.parameters():
            param.requires_grad = False

        in_features = self.model.classifier[2].in_features
        self.model.classifier[2] = nn.Linear(in_features, num_classes)

        if phase == 2:
            for param in self.model.features[7].parameters():
                param.requires_grad = True

        self.criterion = nn.CrossEntropyLoss()

        # ── Metrics ─────────────────────────────────────────────────────
        mc = dict(task='multiclass', num_classes=num_classes)
        self.train_acc   = torchmetrics.Accuracy(**mc)
        self.val_acc     = torchmetrics.Accuracy(**mc)
        self.val_prec    = torchmetrics.Precision(**mc, average='macro')
        self.val_rec     = torchmetrics.Recall(**mc, average='macro')
        self.val_f1      = torchmetrics.F1Score(**mc, average='macro')
        self.val_prec_pc = torchmetrics.Precision(**mc, average='none')
        self.val_rec_pc  = torchmetrics.Recall(**mc, average='none')

        # Accumulate raw preds/labels for the confusion-matrix callback
        self._val_preds  = []
        self._val_labels = []

    # ── Forward ──────────────────────────────────────────────────────────
    def forward(self, x):
        return self.model(x)

    # ── Training ─────────────────────────────────────────────────────────
    def training_step(self, batch, _):
        x, y   = batch
        logits = self(x)
        loss   = self.criterion(logits, y)
        preds  = logits.argmax(dim=1)
        self.train_acc(preds, y)
        self.log('train/loss', loss,           on_step=False, on_epoch=True, prog_bar=True)
        self.log('train/acc',  self.train_acc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    # ── Validation ───────────────────────────────────────────────────────
    def validation_step(self, batch, _):
        x, y   = batch
        logits = self(x)
        loss   = self.criterion(logits, y)
        preds  = logits.argmax(dim=1)

        self.val_acc(preds, y)
        self.val_prec(preds, y)
        self.val_rec(preds, y)
        self.val_f1(preds, y)
        self.val_prec_pc(preds, y)
        self.val_rec_pc(preds, y)

        self._val_preds.append(preds.cpu())
        self._val_labels.append(y.cpu())

        self.log('val/loss',      loss,          on_step=False, on_epoch=True, prog_bar=True)
        self.log('val/acc',       self.val_acc,  on_step=False, on_epoch=True, prog_bar=True)
        self.log('val/precision', self.val_prec, on_step=False, on_epoch=True)
        self.log('val/recall',    self.val_rec,  on_step=False, on_epoch=True)
        self.log('val/f1',        self.val_f1,   on_step=False, on_epoch=True)

    def on_validation_epoch_end(self):
        # Per-class precision & recall (logged to WandB, not prog_bar)
        prec_pc = self.val_prec_pc.compute()
        rec_pc  = self.val_rec_pc.compute()
        for i, name in enumerate(CLASS_NAMES):
            self.log(f'val/precision_{name}', prec_pc[i])
            self.log(f'val/recall_{name}',    rec_pc[i])
        self.val_prec_pc.reset()
        self.val_rec_pc.reset()

    # ── Optimiser + LR scheduler ─────────────────────────────────────────
    def configure_optimizers(self):
        params    = [p for p in self.model.parameters() if p.requires_grad]
        optimizer = optim.AdamW(params, lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=self.hparams.max_epochs,
            eta_min=self.hparams.lr * 0.1,
        )
        return {'optimizer': optimizer,
                'lr_scheduler': {'scheduler': scheduler, 'interval': 'epoch'}}

In [ ]:
class ConfusionMatrixCallback(Callback):
    """Log a colour-coded confusion matrix to WandB after every validation epoch."""

    def __init__(self, class_names: list, phase_name: str):
        self.class_names = class_names
        self.phase_name  = phase_name

    def on_validation_epoch_end(self, trainer, pl_module):
        if not pl_module._val_preds:
            return

        all_preds  = torch.cat(pl_module._val_preds).numpy()
        all_labels = torch.cat(pl_module._val_labels).numpy()
        pl_module._val_preds.clear()
        pl_module._val_labels.clear()

        cm = confusion_matrix(all_labels, all_preds)
        fig, ax = plt.subplots(figsize=(7, 6))
        sns.heatmap(
            cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=self.class_names,
            yticklabels=self.class_names,
            ax=ax,
        )
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
        ax.set_title(
            f'Confusion Matrix – {self.phase_name}  epoch {trainer.current_epoch + 1}'
        )
        plt.tight_layout()

        trainer.logger.experiment.log({
            f'confusion_matrix/{self.phase_name}': wandb.Image(fig),
            'epoch': trainer.current_epoch,
        })
        plt.close(fig)


class SamplePredictionCallback(Callback):
    """Log a grid of validation images with GT vs predicted labels every N epochs."""

    _IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
    _IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

    def __init__(self, val_dataset, class_names, num_samples=8, every_n_epochs=2):
        self.class_names    = class_names
        self.num_samples    = num_samples
        self.every_n_epochs = every_n_epochs
        indices = torch.randperm(len(val_dataset))[:num_samples].tolist()
        self._samples = [val_dataset[i] for i in indices]

    def on_validation_epoch_end(self, trainer, pl_module):
        if (trainer.current_epoch + 1) % self.every_n_epochs != 0:
            return

        pl_module.eval()
        images, gt_labels, pred_labels = [], [], []
        with torch.no_grad():
            for img, label in self._samples:
                logit = pl_module(img.unsqueeze(0).to(pl_module.device))
                pred  = logit.argmax(dim=1).item()
                denorm = (img.cpu() * self._IMAGENET_STD[:, None, None]
                          + self._IMAGENET_MEAN[:, None, None]).clamp(0, 1)
                images.append(denorm)
                gt_labels.append(label)
                pred_labels.append(pred)

        cols = 4
        rows = (self.num_samples + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
        axes = axes.flatten()
        for ax, img, lbl, pred in zip(axes, images, gt_labels, pred_labels):
            ax.imshow(img.permute(1, 2, 0).numpy())
            color = 'green' if pred == lbl else 'red'
            ax.set_title(
                f'GT: {self.class_names[lbl]}\nPred: {self.class_names[pred]}',
                color=color, fontsize=8,
            )
            ax.axis('off')
        for ax in axes[len(images):]:
            ax.axis('off')
        plt.suptitle(f'Sample Predictions – epoch {trainer.current_epoch + 1}')
        plt.tight_layout()

        trainer.logger.experiment.log({
            'sample_predictions': wandb.Image(fig),
            'epoch': trainer.current_epoch,
        })
        plt.close(fig)
        pl_module.train()

In [ ]:
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WEAVE_DISABLED"] = "true"
wandb.login(key=WANDB_API_KEY)

run = wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_NAME,
    config={
        'architecture'  : 'ConvNeXt-Tiny',
        'num_classes'   : NUM_CLASSES,
        'classes'       : CLASS_NAMES,
        'batch_size'    : BATCH_SIZE,
        'epochs_phase1' : EPOCHS_P1,
        'epochs_phase2' : EPOCHS_P2,
        'lr_phase1'     : LR_P1,
        'lr_phase2'     : LR_P2,
        'weight_decay'  : WEIGHT_DECAY,
    },
)

# Use epoch as the x-axis for every metric in the WandB dashboard
wandb.define_metric("epoch")
wandb.define_metric("*", step_metric="epoch")

wandb_logger = WandbLogger(experiment=run, log_model=False)

# ── Phase 1: train classifier head only ──────────────────────────────────
print("=" * 55)
print("Phase 1 – classifier head, backbone frozen")
print("=" * 55)

phase1_model = ConvNextClassifier(
    num_classes=NUM_CLASSES, phase=1, lr=LR_P1,
    weight_decay=WEIGHT_DECAY, max_epochs=EPOCHS_P1,
)

callbacks_p1 = [
    ModelCheckpoint(
        monitor='val/acc', mode='max', save_top_k=1, verbose=True,
        filename='phase1-{epoch:02d}-{val_acc:.4f}',
    ),
    LearningRateMonitor(logging_interval='epoch'),
    ConfusionMatrixCallback(CLASS_NAMES, phase_name='phase1'),
    SamplePredictionCallback(val_dataset, CLASS_NAMES, num_samples=8, every_n_epochs=2),
]

trainer1 = pl.Trainer(
    max_epochs=EPOCHS_P1,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    logger=wandb_logger,
    callbacks=callbacks_p1,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

trainer1.fit(phase1_model, train_loader, val_loader)
print(f"\nPhase 1 – best val/acc: {callbacks_p1[0].best_model_score:.4f}")

In [ ]:
# ── Phase 2: fine-tune classifier + last backbone stage ──────────────────
print("=" * 55)
print("Phase 2 – fine-tuning last stage + classifier")
print("=" * 55)

# Load best Phase 1 weights into a new Phase 2 module
phase2_model = ConvNextClassifier(
    num_classes=NUM_CLASSES, phase=2, lr=LR_P2,
    weight_decay=WEIGHT_DECAY, max_epochs=EPOCHS_P2,
)
best_p1_ckpt = torch.load(callbacks_p1[0].best_model_path, map_location='cpu')
phase2_model.load_state_dict(best_p1_ckpt['state_dict'])

callbacks_p2 = [
    ModelCheckpoint(
        monitor='val/acc', mode='max', save_top_k=1, verbose=True,
        filename='phase2-{epoch:02d}-{val_acc:.4f}',
    ),
    LearningRateMonitor(logging_interval='epoch'),
    EarlyStopping(monitor='val/acc', mode='max', patience=6, verbose=True),
    ConfusionMatrixCallback(CLASS_NAMES, phase_name='phase2'),
    SamplePredictionCallback(val_dataset, CLASS_NAMES, num_samples=8, every_n_epochs=2),
]

trainer2 = pl.Trainer(
    max_epochs=EPOCHS_P2,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    devices=1,
    logger=wandb_logger,
    callbacks=callbacks_p2,
    log_every_n_steps=1,
    enable_progress_bar=True,
)

trainer2.fit(phase2_model, train_loader, val_loader)
print(f"\nPhase 2 – best val/acc: {callbacks_p2[0].best_model_score:.4f}")

# ── Final classification report on best Phase 2 model ────────────────────
print("\n── Final Evaluation ─────────────────────────────────────────")
best_model = ConvNextClassifier.load_from_checkpoint(
    callbacks_p2[0].best_model_path, phase=2,
)
best_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
best_model = best_model.to(device)

all_preds, all_labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        preds = best_model(x.to(device)).argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.tolist())

report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
print(report)

# Log the report as a WandB text artifact
wandb.run.summary['final_classification_report'] = report

# ── Upload best model as WandB Artifact ──────────────────────────────────
artifact = wandb.Artifact(
    name=f"convnext-tiny-{wandb.run.id}",
    type='model',
    metadata={'phase': 2, 'classes': CLASS_NAMES},
)
artifact.add_file(callbacks_p2[0].best_model_path)
wandb.log_artifact(artifact)
print(f"\nBest model uploaded: {callbacks_p2[0].best_model_path}")

wandb.finish()

Phase 2 – fine-tuning last stage + classifier


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
C:\Users\ndvam\anaconda3\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:654: Checkpoint directory .\lightning_logs\aheawtx4\checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type                | Params | Mode 
------------------------------------------------------------
0 | model       | ConvNeXt            | 27.8 M | train
1 | criterion   | CrossEntropyLoss    | 0      | train
2 | train_acc   | MulticlassAccuracy  | 0      | train
3 | val_acc     | MulticlassAccuracy  | 0      | train
4 | val_prec    | MulticlassPrecision | 0      | train
5 | val_rec     | MulticlassRecall    | 0      | train
6 | val_f1      | MulticlassF1Score   | 0      | train
7 | val_prec_pc | MulticlassPrecision | 0      | train
8 | val_rec_pc  | MulticlassRecall    | 0      | train
--------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 13, global step 1162: 'val/acc' was not in top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 14, global step 1245: 'val/acc' was not in top 1
`Trainer.fit` stopped: `max_epochs=15` reached.



Phase 2 – best val/acc: 0.8566

── Final Evaluation ─────────────────────────────────────────
              precision    recall  f1-score   support

  background       0.91      0.86      0.88       439
        fish       0.81      0.85      0.83       386
partial_fish       0.85      0.86      0.86       793

    accuracy                           0.86      1618
   macro avg       0.86      0.85      0.86      1618
weighted avg       0.86      0.86      0.86      1618


Best model uploaded: .\lightning_logs\aheawtx4\checkpoints\phase2-epoch=10-val_acc=0.0000.ckpt
